# Lab 04 — SCD Type 2 Product Dimension

This notebook builds a **historical product dimension** from the validated Silver transaction table and maintains it with Delta Lake `MERGE`.

SCD Type 2 preserves attribute history. When a tracked value changes, the current record is closed and a new current version is inserted. This makes it possible to answer both **what is true now?** and **what was true at a previous point in time?**

## Objectives

- Derive one deterministic product record per `stock_code`.
- Create version-level and entity-level surrogate keys.
- Seed version 1 without duplicating existing products.
- Generate a controlled product-change batch.
- Close changed current rows and insert replacement versions.
- Validate current-row uniqueness, version sequencing, and effective-date ranges.
- Replay the same batch and prove idempotency.


## 1. Load shared configuration

The shared configuration supplies the catalog, schema, volume paths, table names, batch identifier, and guarded reset option used throughout Lab 4.

In [0]:
%run ./lab04_00_config

# Lab 04 — Runtime Configuration

This notebook is intentionally **DDL-free**.

It:
- defines runtime widgets and validates their values;
- builds reusable paths and table names;
- loads the version-controlled YAML data contracts;
- exposes one runtime-selected contract plus the v1/v2 references required by the schema-governance demonstrations.

It does **not** create catalogs, schemas, volumes, folders, or tables.

Run `lab04_00_setup` manually once when the Lab 4 structure must be created or verified. Production Job tasks may `%run ./lab04_00_config` safely.


runtime_selection,contract_name,yaml_version,governance_status,column_count,supersedes,runtime_selected
v1,online_retail,1,active,8,null,true


Runtime configuration ready: dbr_dev.parvinbadalov
Volume: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality
Source workbook: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/Online Retail.xlsx
Runtime contract: online_retail v1 (governance status: active)
Schema policy: fail


In [0]:
import sys

from pyspark.sql import functions as F
from pyspark.sql.window import Window

lab04_root = (
    "/Workspace/Users/parvinbadalov@yahoo.com/"
    "Databricks-Academy-Lakehouse/labs/lab_04_silver_quality"
)

if lab04_root not in sys.path:
    sys.path.append(lab04_root)

from src.merge_utils import (
    add_record_hash,
    apply_scd_type2_plan,
    classify_scd2_changes,
)

product_scd2_table = table_names["product_scd2"]

scd2_change_batch_id = f"{batch_id}_scd2_change"
scd2_change_path = (
    f"{paths['scd_changes']}/type2/batch_id={scd2_change_batch_id}"
)

scd2_seed_plan_path = (
    f"{paths['scd_changes']}/type2_seed_plan/batch_id={batch_id}"
)

scd2_plan_path = (
    f"{paths['scd_changes']}/type2_plan/batch_id={scd2_change_batch_id}"
)

print(f"Silver source: {silver_table}")
print(f"SCD Type 2 target: {product_scd2_table}")
print(f"Controlled change path: {scd2_change_path}")
print("Reusable SCD2 module: src.merge_utils")


Silver source: dbr_dev.parvinbadalov.lab04_silver_transactions
SCD Type 2 target: dbr_dev.parvinbadalov.lab04_product_scd2
Controlled change path: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/test_data/scd_changes/type2/batch_id=initial_scd2_change
Reusable SCD2 module: src.merge_utils


## 2. Verify the Silver transaction source

The dimension is derived only from the analytics-ready Silver table produced by notebook 04. The notebook stops early when the source table is missing, empty, or does not satisfy the expected interface.

In [0]:
if not spark.catalog.tableExists(silver_table):
    raise RuntimeError(
        f"Required Silver table does not exist: {silver_table}. "
        "Run lab04_04_silver_merge.ipynb first."
    )

silver_transactions_df = spark.table(silver_table)
silver_transaction_count = silver_transactions_df.count()
if silver_transaction_count == 0:
    raise ValueError(f"Silver transaction table is empty: {silver_table}")

required_source_columns = {
    "transaction_line_id", "invoice_no", "stock_code", "description",
    "unit_price", "invoice_timestamp", "source_batch_id",
    "silver_updated_at",
}
missing_source_columns = sorted(required_source_columns - set(silver_transactions_df.columns))
if missing_source_columns:
    raise AssertionError(f"Silver source is missing columns: {missing_source_columns}")

print(f"✅ Silver source ready with {silver_transaction_count:,} transactions.")


✅ Silver source ready with 315,101 transactions.


## 3. Derive one baseline record per product

Products occur in many transaction rows, so the latest observation is selected with deterministic tie-breakers.

- `product_sk` identifies the product entity and remains stable across versions.
- `source_record_hash` represents the tracked Type 2 attributes.
- `product_version_sk` will identify one specific historical version.

The hash prevents unnecessary history records when a source row is replayed unchanged.

In [0]:
latest_product_window = (
    Window.partitionBy("stock_code")
    .orderBy(
        F.col("invoice_timestamp").desc(),
        F.col("silver_updated_at").desc(),
        F.col("transaction_line_id").desc(),
    )
)

latest_product_rows_df = (
    silver_transactions_df
    .filter(
        F.col("stock_code").isNotNull()
        & (F.length(F.trim("stock_code")) > 0)
    )
    .withColumn(
        "_product_rank",
        F.row_number().over(latest_product_window),
    )
    .filter(F.col("_product_rank") == 1)
)

product_source_df = (
    latest_product_rows_df
    .select(
        F.sha2(F.col("stock_code"), 256).alias("product_sk"),
        F.col("stock_code"),
        F.col("description"),
        F.col("unit_price")
        .cast("decimal(18,4)")
        .alias("latest_observed_price"),
        F.col("invoice_no").alias("source_invoice_no"),
        F.col("transaction_line_id").alias(
            "source_transaction_line_id"
        ),
        F.col("invoice_timestamp").alias(
            "source_event_timestamp"
        ),
        F.col("source_batch_id"),
    )
)

product_source_df = add_record_hash(
    product_source_df,
    [
        "stock_code",
        "description",
        "latest_observed_price",
    ],
    output_column="source_record_hash",
)

product_source_count = product_source_df.count()
product_source_distinct = (
    product_source_df.select("stock_code").distinct().count()
)

if (
    product_source_count == 0
    or product_source_count != product_source_distinct
):
    raise AssertionError(
        f"Product source key validation failed: "
        f"rows={product_source_count}, "
        f"distinct stock codes={product_source_distinct}."
    )

print(f"Derived {product_source_count:,} unique products.")
display(product_source_df.orderBy("stock_code").limit(30))


Derived 3,628 unique products.


product_sk,stock_code,description,latest_observed_price,source_invoice_no,source_transaction_line_id,source_event_timestamp,source_batch_id,source_record_hash
56acd2f85500d246e0851f77035f8b4414a57c4f31735c8f37c9cec0f3739d47,10002,INFLATABLE POLITICAL GLOBE,0.8500,550452,5a5ad332b65c0ef0ba84a9fea5d9465f467333ac50995e57fd61d316de648084,2011-04-18T12:56:00.000Z,initial,101f81f320fbbde205a982f4d76faae2ec9873a1e60a2b11c3ad29666f39e061
2ab4e55c7fdf332c78e4f36966a981f452ffb6758026085a4ad9df565a1df82d,10080,GROOVY CACTUS INFLATABLE,0.3900,577773,574af94ff0b2803d98464b20759fa5e04e2750a5067e962ea300718d523ee275,2011-11-21T15:57:00.000Z,initial,362dafe771dbedd66c5328ea4b72748759622674b15e74bf2b0d497e077c7e9c
801aaa199f1030bd990a6bd8da7e1bd76f1461bee56880924ccbd49f1166178b,10120,DOGGY RUBBER,0.2100,580502,34c037f99e74e8dc9b0ab51a0910044bf6a0d4502c6677d44d33169ccd1b3a5b,2011-12-04T13:15:00.000Z,initial,2bc8de476d7e9e68ae6f5c83a14a960a3dc9c5f530b6cf156f92853ecc63a5d1
c603961ad4629b9a249a64757df4c2d63249ad4babd6c908a97bcc679368ee8d,10123C,HEARTS WRAPPING TAPE,0.6500,548491,43881e426937a68d6c103c10d9f652c92e1d9342ac7195ba06a28d20418ce3e4,2011-03-31T13:14:00.000Z,initial,f90770ecaa1c4629e4bd8c1d287e02049ca21d7bb26cf00ffdbe481ff2ecb994
6a84538adbe548d147f2d20d9e6df55d9ed7ff53e4a4326f695eccb031fc13ab,10124A,SPOTS ON RED BOOKCOVER TAPE,0.4200,574686,d49304baf4e6d457d5ea5afc5d0921b5e376930e75e239b51a26acab784661e7,2011-11-06T13:00:00.000Z,initial,15ca5847e977cb6a108ef084967826d56aff15672373d1d7cb4f55e64a91b5bf
63d621285bb94218c6676555283e19fa7bb61719ec56c2416255164a73fb88ed,10124G,ARMY CAMO BOOKCOVER TAPE,0.4200,574686,ef20dd2506d762c7113ddd959d29fdc1d5a0e060a121e305d7fdb151dc2e4f0c,2011-11-06T13:00:00.000Z,initial,2738f8d05e6885a4e8bae18914c3bb439da25e4396d86ebe881abb4fd5772636
7cb3fc848fa0e795b7b15cbef125090b271d93b387645020e9788219e2fda761,10125,MINI FUNKY DESIGN TAPES,0.8500,581494,e95fc1f9f1bb16338a42fbecdeb751fec3d47f83a262c4b9ff831d48bc579a06,2011-12-09T10:13:00.000Z,initial,8bf93d52c48e03395788df46878986406141be8d53a02dbb6650e41be8585603
7d686424482e81086e0e399c1f1bd9173551b38dcf2221902c9c067876fd4e56,10133,COLOURING PENCILS BROWN TUBE,0.4200,565541,ec543ed93e763d4d54cf3d58125d9a6d07a857b9e1e6939df3d6d1898ba5c8d6,2011-09-05T12:00:00.000Z,initial,2ee072ab0fd7dc2c222e770aece8b2cb32b72fa33c8b0379bcea19e2de5a0938
9190a41c01288f3725256e28f95cfd7679d185ad55d42ad5a1e2a5d7c2b091e3,10135,COLOURING PENCILS BROWN TUBE,2.4600,580727,6cd902c1d48299d517ecd3f4f0315e779311670d8258df1c3e1b20472e163c37,2011-12-05T17:17:00.000Z,initial,ae4fc1c31cf134345d55f99f2a30caed21832329ad11e399a98d42821a63f65c
72b209306c1a1031b9b3dbec63cf58b0beabbf3e3f0a40c63ef5df093b62dfb8,11001,ASSTD DESIGN RACING CAR PEN,3.2900,580727,6b08b9cf60eae708bdf74155f28fc34dce782461ecdbf92da42663b4d4e6e715,2011-12-05T17:17:00.000Z,initial,213f38231320d1e90cf743a4dd2e9f01219ad82544fbb4d1aa4b802466dcb3a7


## 4. Verify the pre-created SCD Type 2 target

The historical dimension structure is created by **`lab04_00_setup`** outside the production Job.

The Job notebook performs no structural DDL. It validates the required columns before seeding or applying historical changes.


In [0]:
if reset_demo_objects:
    raise ValueError(
        "reset_demo_objects=true is not allowed inside the production Job. "
        "Run lab04_00_setup manually for a clean structural rebuild."
    )

if not spark.catalog.tableExists(product_scd2_table):
    raise RuntimeError(
        f"Required SCD Type 2 table does not exist: {product_scd2_table}. "
        "Run lab04_00_setup manually before executing the Job."
    )

required_scd2_columns = {
    "product_version_sk",
    "product_sk",
    "stock_code",
    "description",
    "latest_observed_price",
    "version_number",
    "source_invoice_no",
    "source_transaction_line_id",
    "source_event_timestamp",
    "source_record_hash",
    "source_batch_id",
    "effective_from",
    "effective_to",
    "is_current",
    "created_at",
    "updated_at",
}

missing_scd2_columns = sorted(
    required_scd2_columns - set(spark.table(product_scd2_table).columns)
)

if missing_scd2_columns:
    raise AssertionError(
        "Pre-created SCD Type 2 target is missing columns: "
        + ", ".join(missing_scd2_columns)
    )

print(f"✅ Pre-created SCD Type 2 table is ready: {product_scd2_table}")


✅ Pre-created SCD Type 2 table is ready: dbr_dev.parvinbadalov.lab04_product_scd2


## 5. Seed version 1 idempotently

The baseline seed exists only to create **missing product keys**.

This is important for SCD Type 2: after a product has historical versions, rerunning the notebook must **not** compare the original baseline to the latest current version and interpret that difference as a new change.

Therefore:

1. classify the baseline against the current SCD2 target;
2. keep only `INSERT` rows for the seed;
3. ignore `UPDATE` classifications during baseline seeding;
4. apply real historical changes only in the controlled-change section.

This makes the seed safe on both a clean table and an already-populated historical dimension.


In [0]:
baseline_incoming_df = (
    product_source_df
    .withColumn(
        "change_effective_at",
        F.col("source_event_timestamp"),
    )
)

baseline_classification_df = classify_scd2_changes(
    baseline_incoming_df,
    product_scd2_table,
    business_key="stock_code",
    hash_column="source_record_hash",
)

# Show how the complete baseline compares with the current target.
baseline_classification_summary_df = (
    baseline_classification_df
    .groupBy("merge_action")
    .agg(F.count("*").alias("rows"))
    .orderBy("merge_action")
)

# IMPORTANT:
# Baseline seeding is INSERT-ONLY.
# Existing keys may be classified UPDATE because their current SCD2 version
# legitimately differs from the original baseline after previous lab runs.
# Those rows must not be closed/re-versioned by the seed step.
baseline_seed_df = baseline_classification_df.filter(
    F.col("merge_action") == "INSERT"
)

expected_seed_inserts = baseline_seed_df.count()

target_count_before_seed = spark.table(
    product_scd2_table
).count()

scd2_insert_values = {
    "product_version_sk": (
        "sha2(concat_ws('||', source.stock_code, "
        "source.source_record_hash, cast(source.version_number as string)), 256)"
    ),
    "product_sk": "source.product_sk",
    "stock_code": "source.stock_code",
    "description": "source.description",
    "latest_observed_price": "source.latest_observed_price",
    "source_invoice_no": "source.source_invoice_no",
    "source_transaction_line_id": "source.source_transaction_line_id",
    "source_event_timestamp": "source.source_event_timestamp",
    "source_record_hash": "source.source_record_hash",
    "source_batch_id": "source.source_batch_id",
    "created_at": "current_timestamp()",
}

seed_metrics = apply_scd_type2_plan(
    baseline_seed_df,
    product_scd2_table,
    plan_path=scd2_seed_plan_path,
    business_key="stock_code",
    effective_at_column="change_effective_at",
    insert_values=scd2_insert_values,
    hash_column="source_record_hash",
    version_column="version_number",
    current_column="is_current",
    effective_from_column="effective_from",
    effective_to_column="effective_to",
    updated_at_column="updated_at",
)

target_count_after_seed = spark.table(
    product_scd2_table
).count()

actual_seed_growth = (
    target_count_after_seed - target_count_before_seed
)

if seed_metrics["closed_rows"] != 0:
    raise AssertionError(
        "Baseline seed must be INSERT-only, but rows were prepared for closure."
    )

if seed_metrics["inserted_rows"] != expected_seed_inserts:
    raise AssertionError(
        f"Seed mismatch: expected {expected_seed_inserts} inserts, "
        f"prepared {seed_metrics['inserted_rows']}."
    )

if actual_seed_growth != expected_seed_inserts:
    raise AssertionError(
        f"Seed growth mismatch: expected {expected_seed_inserts}, "
        f"observed {actual_seed_growth}."
    )

display(baseline_classification_summary_df)

print(f"Versions before seed: {target_count_before_seed:,}")
print(f"Missing product keys seeded: {actual_seed_growth:,}")
print(f"Versions after seed: {target_count_after_seed:,}")

existing_baseline_rows = (
    baseline_classification_df.filter(
        F.col("merge_action") != "INSERT"
    ).count()
)

if existing_baseline_rows:
    print(
        f"ℹ️ {existing_baseline_rows:,} baseline products already existed. "
        "They were intentionally left untouched by the INSERT-only seed."
    )

print("✅ Baseline SCD2 seed completed without modifying existing history.")


merge_action,rows
UNCHANGED,3628


Versions before seed: 0
Missing product keys seeded: 3,628
Versions after seed: 3,628
ℹ️ 3,628 baseline products already existed. They were intentionally left untouched by the INSERT-only seed.
✅ Baseline SCD2 seed completed without modifying existing history.


## 6. Generate a deterministic Type 2 change batch

The public workbook is a transaction extract rather than a product-master change feed. To demonstrate history safely, the lab creates three controlled changes in a dedicated test-data folder:

- append ` [SCD2 VERSION 2]` to the description,
- increase the observed price by `2.0000`,
- use a deterministic effective timestamp later than version 1,
- calculate a new tracked-attribute hash.

The original source and Silver tables are not modified.

In [0]:
change_sample_size = min(3, product_source_count)

if change_sample_size == 0:
    raise ValueError(
        "No products are available for the SCD Type 2 change test."
    )

scd2_change_df = (
    product_source_df
    .orderBy("stock_code")
    .limit(change_sample_size)
    .withColumn(
        "description",
        F.concat(
            F.regexp_replace(
                F.coalesce(
                    F.col("description"),
                    F.lit("UNKNOWN PRODUCT"),
                ),
                r" \[SCD2 VERSION 2\]$",
                "",
            ),
            F.lit(" [SCD2 VERSION 2]"),
        ),
    )
    .withColumn(
        "latest_observed_price",
        (
            F.coalesce(
                F.col("latest_observed_price"),
                F.lit(0).cast("decimal(18,4)"),
            )
            + F.lit(2).cast("decimal(18,4)")
        ).cast("decimal(18,4)"),
    )
    .withColumn(
        "source_batch_id",
        F.lit(scd2_change_batch_id),
    )
    .withColumn(
        "change_effective_at",
        F.col("source_event_timestamp")
        + F.expr("INTERVAL 365 DAYS"),
    )
    .drop("source_record_hash")
)

scd2_change_df = add_record_hash(
    scd2_change_df,
    [
        "stock_code",
        "description",
        "latest_observed_price",
    ],
    output_column="source_record_hash",
)

(
    scd2_change_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(scd2_change_path)
)

persisted_change_df = (
    spark.read.format("delta").load(scd2_change_path)
)

if persisted_change_df.count() != change_sample_size:
    raise AssertionError(
        "Persisted SCD Type 2 test batch count does not match "
        "the generated batch."
    )

print(
    f"✅ Controlled Type 2 change batch written with "
    f"{change_sample_size} products."
)
display(persisted_change_df.orderBy("stock_code"))


✅ Controlled Type 2 change batch written with 3 products.


product_sk,stock_code,description,latest_observed_price,source_invoice_no,source_transaction_line_id,source_event_timestamp,source_batch_id,change_effective_at,source_record_hash
56acd2f85500d246e0851f77035f8b4414a57c4f31735c8f37c9cec0f3739d47,10002,INFLATABLE POLITICAL GLOBE [SCD2 VERSION 2],2.8500,550452,5a5ad332b65c0ef0ba84a9fea5d9465f467333ac50995e57fd61d316de648084,2011-04-18T12:56:00.000Z,initial_scd2_change,2012-04-17T12:56:00.000Z,6327e6a0196127c6914dcc0e430d68fe50d9c83948aaeb9caacc105704dcd552
2ab4e55c7fdf332c78e4f36966a981f452ffb6758026085a4ad9df565a1df82d,10080,GROOVY CACTUS INFLATABLE [SCD2 VERSION 2],2.3900,577773,574af94ff0b2803d98464b20759fa5e04e2750a5067e962ea300718d523ee275,2011-11-21T15:57:00.000Z,initial_scd2_change,2012-11-20T15:57:00.000Z,a7115855da193e148fb15b801208f0cca6e1d5b6d4f9adc1abd8ea0edf12b503
801aaa199f1030bd990a6bd8da7e1bd76f1461bee56880924ccbd49f1166178b,10120,DOGGY RUBBER [SCD2 VERSION 2],2.2100,580502,34c037f99e74e8dc9b0ab51a0910044bf6a0d4502c6677d44d33169ccd1b3a5b,2011-12-04T13:15:00.000Z,initial_scd2_change,2012-12-03T13:15:00.000Z,9c64d9ce7427ffd9daefce142a6e419685708a99f1c470402f58484acdf106f2


## 7. Classify incoming rows against current versions

The classifier returns a flat, auditable merge plan:

- `INSERT`: no current product version exists,
- `UPDATE`: a current version exists but the tracked hash changed,
- `UNCHANGED`: the incoming hash already matches the current version.

Only INSERT and UPDATE rows are passed to the write phase.

In [0]:
current_before_change_df = (
    spark.table(product_scd2_table)
    .filter(F.col("is_current"))
    .select(
        F.col("stock_code").alias("_evidence_stock_code"),
        F.col("description").alias("previous_description"),
        F.col("latest_observed_price").alias("previous_price"),
    )
)

change_classification_df = classify_scd2_changes(
    persisted_change_df,
    product_scd2_table,
    business_key="stock_code",
    hash_column="source_record_hash",
    version_column="version_number",
    current_column="is_current",
    effective_from_column="effective_from",
)

change_plan_df = (
    change_classification_df
    .groupBy("merge_action")
    .agg(F.count("*").alias("rows"))
    .orderBy("merge_action")
)

change_plan = {
    row["merge_action"]: row["rows"]
    for row in change_plan_df.collect()
}

expected_updates = change_plan.get("UPDATE", 0)
expected_inserts = change_plan.get("INSERT", 0)
expected_unchanged = change_plan.get("UNCHANGED", 0)

change_evidence_df = (
    change_classification_df.alias("plan")
    .join(
        current_before_change_df.alias("previous"),
        F.col("plan.stock_code")
        == F.col("previous._evidence_stock_code"),
        "left",
    )
    .select(
        F.col("plan.stock_code"),
        F.col("plan.merge_action"),
        F.col("plan.previous_version_number"),
        F.col("previous.previous_description"),
        F.col("plan.description").alias("incoming_description"),
        F.col("previous.previous_price"),
        F.col("plan.latest_observed_price").alias("incoming_price"),
        F.col("plan.change_effective_at"),
    )
    .orderBy("stock_code")
)

display(change_plan_df)
display(change_evidence_df)


merge_action,rows
UPDATE,3


stock_code,merge_action,previous_version_number,previous_description,incoming_description,previous_price,incoming_price,change_effective_at
10002,UPDATE,1,INFLATABLE POLITICAL GLOBE,INFLATABLE POLITICAL GLOBE [SCD2 VERSION 2],0.8500,2.8500,2012-04-17T12:56:00.000Z
10080,UPDATE,1,GROOVY CACTUS INFLATABLE,GROOVY CACTUS INFLATABLE [SCD2 VERSION 2],0.3900,2.3900,2012-11-20T15:57:00.000Z
10120,UPDATE,1,DOGGY RUBBER,DOGGY RUBBER [SCD2 VERSION 2],0.2100,2.2100,2012-12-03T13:15:00.000Z


## 8. Apply the reusable two-step SCD Type 2 pipeline

`src.merge_utils.apply_scd_type2_plan()` freezes the classified plan to Delta before changing the target.

It then:

1. closes changed current versions;
2. inserts the next current version;
3. leaves unchanged rows untouched.

This centralizes the SCD2 write pattern so notebooks and tests do not maintain separate implementations.


In [0]:
target_count_before_change = spark.table(
    product_scd2_table
).count()

change_metrics = apply_scd_type2_plan(
    change_classification_df,
    product_scd2_table,
    plan_path=scd2_plan_path,
    business_key="stock_code",
    effective_at_column="change_effective_at",
    insert_values=scd2_insert_values,
    hash_column="source_record_hash",
    version_column="version_number",
    current_column="is_current",
    effective_from_column="effective_from",
    effective_to_column="effective_to",
    updated_at_column="updated_at",
)

closed_count = change_metrics["closed_rows"]
inserted_version_count = change_metrics["inserted_rows"]

target_count_after_change = spark.table(
    product_scd2_table
).count()

actual_history_growth = (
    target_count_after_change - target_count_before_change
)

if closed_count != expected_updates:
    raise AssertionError(
        f"Expected {expected_updates} closed rows, "
        f"prepared {closed_count}."
    )

if inserted_version_count != expected_updates + expected_inserts:
    raise AssertionError(
        f"Expected {expected_updates + expected_inserts} new versions, "
        f"prepared {inserted_version_count}."
    )

if actual_history_growth != expected_updates + expected_inserts:
    raise AssertionError(
        f"History growth mismatch: "
        f"expected {expected_updates + expected_inserts}, "
        f"observed {actual_history_growth}."
    )

print(f"Current versions closed: {closed_count:,}")
print(
    f"Replacement/new versions inserted: "
    f"{inserted_version_count:,}"
)
print(f"Historical row growth: {actual_history_growth:,}")
print(
    "✅ Two-step SCD Type 2 pipeline completed through "
    "src.merge_utils.apply_scd_type2_plan()."
)


Current versions closed: 3
Replacement/new versions inserted: 3
Historical row growth: 3
✅ Two-step SCD Type 2 pipeline completed through src.merge_utils.apply_scd_type2_plan().


## 9. Inspect controlled version history

A clean first execution creates two versions for each controlled product: the original historical row and the new current row.

However, an SCD Type 2 table is a historical table by design. If this lab has been executed previously, a controlled product may legitimately already contain more than two historical versions.

Therefore this evidence step does **not** assume that the whole dimension is empty before the run. It verifies that:

- every controlled product has at least two versions;
- every controlled product has exactly one current row;
- the two most recent versions can be displayed clearly as the before/after evidence for this run.

The full temporal and version-integrity checks in the next section still validate the complete history.


In [0]:
controlled_stock_codes_df = (
    persisted_change_df
    .select("stock_code")
    .distinct()
)

controlled_history_df = (
    spark.table(product_scd2_table)
    .join(
        controlled_stock_codes_df,
        "stock_code",
        "inner",
    )
)

controlled_history_summary_df = (
    controlled_history_df
    .groupBy("stock_code")
    .agg(
        F.count("*").alias("version_rows"),
        F.countDistinct("version_number").alias(
            "distinct_version_numbers"
        ),
        F.sum(
            F.col("is_current").cast("int")
        ).alias("current_rows"),
        F.max("version_number").alias("latest_version_number"),
    )
    .orderBy("stock_code")
)

insufficient_history_count = (
    controlled_history_summary_df
    .filter(F.col("version_rows") < 2)
    .count()
)

invalid_current_count = (
    controlled_history_summary_df
    .filter(F.col("current_rows") != 1)
    .count()
)

if insufficient_history_count:
    raise AssertionError(
        f"{insufficient_history_count} controlled products have fewer "
        "than two SCD Type 2 versions."
    )

if invalid_current_count:
    raise AssertionError(
        f"{invalid_current_count} controlled products do not have exactly "
        "one current SCD Type 2 row."
    )

# Show the two most recent versions for each controlled product.
evidence_window = (
    Window
    .partitionBy("stock_code")
    .orderBy(
        F.col("version_number").desc(),
        F.col("effective_from").desc(),
    )
)

history_evidence_df = (
    controlled_history_df
    .withColumn(
        "_evidence_rank",
        F.row_number().over(evidence_window),
    )
    .filter(F.col("_evidence_rank") <= 2)
    .select(
        "stock_code",
        "version_number",
        "description",
        "latest_observed_price",
        "effective_from",
        "effective_to",
        "is_current",
        "source_batch_id",
        "product_version_sk",
    )
    .orderBy("stock_code", "version_number")
)

display(controlled_history_summary_df)
display(history_evidence_df)

extra_history_rows = (
    controlled_history_df.count()
    - (change_sample_size * 2)
)

if extra_history_rows > 0:
    print(
        "ℹ️ Existing SCD2 history detected: controlled products contain "
        f"{extra_history_rows} rows beyond a clean two-version demo. "
        "This is expected when the historical dimension was populated by "
        "earlier lab runs."
    )

print(
    "✅ Controlled SCD2 history validated. "
    "The two most recent versions are displayed as evidence."
)


stock_code,version_rows,distinct_version_numbers,current_rows,latest_version_number
10002,2,2,1,2
10080,2,2,1,2
10120,2,2,1,2


stock_code,version_number,description,latest_observed_price,effective_from,effective_to,is_current,source_batch_id,product_version_sk
10002,1,INFLATABLE POLITICAL GLOBE,0.8500,2011-04-18T12:56:00.000Z,2012-04-17T12:56:00.000Z,false,initial,669f20057fe869415997c764f21c1d86eaad0d907148f5fd9aa936d3b0dcc2ad
10002,2,INFLATABLE POLITICAL GLOBE [SCD2 VERSION 2],2.8500,2012-04-17T12:56:00.000Z,null,true,initial_scd2_change,a793648b83d43ad36aab771e882c3eca79c793428204784776ec73c21ba5496b
10080,1,GROOVY CACTUS INFLATABLE,0.3900,2011-11-21T15:57:00.000Z,2012-11-20T15:57:00.000Z,false,initial,5aa9fbfcbd923cd3fc27de2899fdef5eddfafd77e62deda5e92a9ba27ec3b886
10080,2,GROOVY CACTUS INFLATABLE [SCD2 VERSION 2],2.3900,2012-11-20T15:57:00.000Z,null,true,initial_scd2_change,88037b9935b050491704ae48dcdbd49a2e549c368f22d764fe5a8a67c470d873
10120,1,DOGGY RUBBER,0.2100,2011-12-04T13:15:00.000Z,2012-12-03T13:15:00.000Z,false,initial,234616d5b3595e6dcc868385baef08c3854295e60be0852e50e0ac0684f4b9a3
10120,2,DOGGY RUBBER [SCD2 VERSION 2],2.2100,2012-12-03T13:15:00.000Z,null,true,initial_scd2_change,1e850690f4acd613c93440d7da7d881449db649963a6c25a6150c1fd1cba6e44


✅ Controlled SCD2 history validated. The two most recent versions are displayed as evidence.


## 10. Validate temporal and dimensional quality

The checks below enforce the core SCD Type 2 invariants:

- one current row per `stock_code`,
- unique version surrogate keys,
- unique version numbers within a product,
- current rows have no end timestamp,
- historical rows have a valid end timestamp,
- adjacent versions do not overlap and share a precise boundary.

In [0]:
scd2_df = spark.table(product_scd2_table)

current_key_violations = (
    scd2_df.filter(F.col("is_current"))
    .groupBy("stock_code")
    .count()
    .filter(F.col("count") != 1)
    .count()
)
duplicate_version_keys = (
    scd2_df.groupBy("product_version_sk")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
duplicate_version_numbers = (
    scd2_df.groupBy("stock_code", "version_number")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
invalid_current_end_dates = scd2_df.filter(
    F.col("is_current") & F.col("effective_to").isNotNull()
).count()
invalid_historical_end_dates = scd2_df.filter(
    (~F.col("is_current"))
    & (
        F.col("effective_to").isNull()
        | (F.col("effective_to") <= F.col("effective_from"))
    )
).count()

version_window = Window.partitionBy("stock_code").orderBy("effective_from", "version_number")
temporal_check_df = (
    scd2_df
    .withColumn("next_effective_from", F.lead("effective_from").over(version_window))
    .withColumn(
        "invalid_boundary",
        F.when(
            F.col("next_effective_from").isNotNull(),
            F.col("effective_to").isNull()
            | (F.col("effective_to") != F.col("next_effective_from")),
        ).otherwise(F.col("effective_to").isNotNull()),
    )
)
invalid_temporal_boundaries = temporal_check_df.filter(F.col("invalid_boundary")).count()

quality_results = {
    "current_key_violations": current_key_violations,
    "duplicate_version_keys": duplicate_version_keys,
    "duplicate_version_numbers": duplicate_version_numbers,
    "invalid_current_end_dates": invalid_current_end_dates,
    "invalid_historical_end_dates": invalid_historical_end_dates,
    "invalid_temporal_boundaries": invalid_temporal_boundaries,
}
if any(value != 0 for value in quality_results.values()):
    raise AssertionError(f"SCD Type 2 quality checks failed: {quality_results}")

quality_results_df = spark.createDataFrame(
    [(name, value) for name, value in quality_results.items()],
    ["quality_rule", "violation_count"],
)
display(quality_results_df)
print("✅ SCD Type 2 temporal and dimensional quality checks passed.")


quality_rule,violation_count
current_key_violations,0
duplicate_version_keys,0
duplicate_version_numbers,0
invalid_current_end_dates,0
invalid_historical_end_dates,0
invalid_temporal_boundaries,0


✅ SCD Type 2 temporal and dimensional quality checks passed.


## 11. Replay the same change batch

The plan is recalculated after the first Type 2 load. The current rows now contain the incoming hashes, so all controlled records must be classified as unchanged. Reapplying the two-step function must close zero rows, insert zero versions, and preserve the target count.

In [0]:
count_before_replay = spark.table(
    product_scd2_table
).count()

replay_plan_df = classify_scd2_changes(
    persisted_change_df,
    product_scd2_table,
    business_key="stock_code",
    hash_column="source_record_hash",
    version_column="version_number",
    current_column="is_current",
    effective_from_column="effective_from",
)

replay_action_df = (
    replay_plan_df
    .groupBy("merge_action")
    .agg(F.count("*").alias("rows"))
    .orderBy("merge_action")
)

pending_replay_changes = replay_plan_df.filter(
    F.col("merge_action").isin("INSERT", "UPDATE")
).count()

replay_metrics = apply_scd_type2_plan(
    replay_plan_df,
    product_scd2_table,
    plan_path=scd2_plan_path,
    business_key="stock_code",
    effective_at_column="change_effective_at",
    insert_values=scd2_insert_values,
    hash_column="source_record_hash",
    version_column="version_number",
    current_column="is_current",
    effective_from_column="effective_from",
    effective_to_column="effective_to",
    updated_at_column="updated_at",
)

replay_closed_count = replay_metrics["closed_rows"]
replay_inserted_count = replay_metrics["inserted_rows"]

count_after_replay = spark.table(product_scd2_table).count()

if pending_replay_changes != 0:
    raise AssertionError(
        f"Replay still contained {pending_replay_changes} pending changes."
    )

if replay_closed_count != 0 or replay_inserted_count != 0:
    raise AssertionError(
        f"Replay wrote data: "
        f"closed={replay_closed_count}, "
        f"inserted={replay_inserted_count}."
    )

if count_after_replay != count_before_replay:
    raise AssertionError(
        f"Replay changed target count: "
        f"before={count_before_replay}, "
        f"after={count_after_replay}."
    )

display(replay_action_df)
print(
    "✅ Idempotency passed through reusable SCD2 helpers: "
    "replay closed 0 rows and inserted 0 versions."
)
print(f"Historical dimension remains at {count_after_replay:,} rows.")


merge_action,rows
UNCHANGED,3


✅ Idempotency passed through reusable SCD2 helpers: replay closed 0 rows and inserted 0 versions.
Historical dimension remains at 3,631 rows.


## 12. Review Delta history and final results

Delta history provides operational evidence for both phases of Type 2 processing. The summary records the number of entity keys, historical versions, controlled changes, and replay outcome.

In [0]:
scd2_history_df = spark.sql(f"DESCRIBE HISTORY {product_scd2_table}")
display(
    scd2_history_df.select(
        "version", "timestamp", "operation", "operationParameters", "operationMetrics",
    ).orderBy(F.col("version").desc()).limit(12)
)

final_dimension_df = spark.table(product_scd2_table)
final_results_df = spark.createDataFrame(
    [
        ("silver_transaction_rows", silver_transaction_count),
        ("unique_product_entities", final_dimension_df.select("stock_code").distinct().count()),
        ("total_product_versions", final_dimension_df.count()),
        ("current_product_versions", final_dimension_df.filter(F.col("is_current")).count()),
        ("historical_product_versions", final_dimension_df.filter(~F.col("is_current")).count()),
        ("controlled_change_rows", change_sample_size),
        ("versions_inserted_this_change", inserted_version_count),
        ("replay_pending_changes", pending_replay_changes),
    ],
    ["validation", "result"],
)
display(final_results_df)
print("✅ SCD Type 2 notebook completed successfully.")


version,timestamp,operation,operationParameters,operationMetrics
25,2026-08-10T21:15:10.000Z,MERGE,"Map(predicate -> [""((stock_code#63262 = stock_code#63043) AND is_current#63273)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1116, materializeSourceTimeMs -> 7, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 0, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1083)"
24,2026-08-10T21:15:08.000Z,MERGE,"Map(predicate -> [""((stock_code#63114 = stock_code#63043) AND is_current#63125)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""NOT (source_record_hash#63121 <=> source_record_hash#63051)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 297, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 265, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 0, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 0)"
23,2026-08-10T21:14:53.000Z,MERGE,"Map(predicate -> [""((stock_code#61325 = stock_code#60308) AND is_current#61336)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 6325, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1743, materializeSourceTimeMs -> 7, numTargetRowsInserted -> 3, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 3, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1703)"
22,2026-08-10T21:14:50.000Z,MERGE,"Map(predicate -> [""((stock_code#60387 = stock_code#60308) AND is_current#60398)""], clusterBy -> [], matchedPredicates -> [{""predicate"":""NOT (source_record_hash#60394 <=> source_record_hash#60316)"",""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 7381, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 3, executionTimeMs -> 3539, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 974, numTargetRowsUpdated -> 3, numOutputRows -> 3, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 1, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2532)"
21,2026-08-10T21:14:33.000Z,MERGE,

validation,result
silver_transaction_rows,315101
unique_product_entities,3628
total_product_versions,3631
current_product_versions,3628
historical_product_versions,3
controlled_change_rows,3
versions_inserted_this_change,3
replay_pending_changes,0


✅ SCD Type 2 notebook completed successfully.


## Evidence to capture

Save screenshots of:

1. the unique product source preview,
2. the INSERT / UPDATE / UNCHANGED plan,
3. the reusable two-step pipeline counts,
4. the controlled-history summary plus the two most recent versions,
5. the zero-violation temporal/dimensional quality results,
6. the idempotent replay message,
7. Delta history and the final summary.

## Next notebook

Continue with **`lab04_07_scd_comparison.ipynb`**. It compares Type 1 and Type 2 behavior side by side and validates their different history semantics.
